In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [ ]:
# 엑셀파일
d=pd.read_excel('Overview.xlsx')

In [ ]:
d

In [ ]:
# TripA01~TripA32까지
for i in range(1, 33):
    file_name = f"TripA{i:02d}.csv"
    globals()[f"dfA{i}"] = pd.read_csv(file_name, encoding="ISO-8859-1", delimiter=";")

    # Time [s] 컬럼을 datetime 형식으로 변환
    globals()[f"dfA{i}"]['Time [s]'] = pd.to_datetime(globals()[f"dfA{i}"]['Time [s]'], unit='s', origin='unix')
    # time 컬럼을 5초 단위로 묶기
    globals()[f"dfA{i}"] = globals()[f"dfA{i}"].set_index('Time [s]').resample('5S').mean().reset_index()
    # 필요시 Time [s] 컬럼 다시 생성
    globals()[f"dfA{i}"]['Time [s]'] = (globals()[f"dfA{i}"]['Time [s]'].astype(np.int64) // 10**9).astype(int)
    
# dfA11 결측치 제거
dfA11=dfA11.drop(columns=["Unnamed: 23"], errors="ignore")

# dfA로 통합

# 공통된 column 확인 (A)
import pandas as pd
# 공통 컬럼을 저장할 변수 (초기값을 None으로 설정)
common_columns_A = None
for i in range(1, 33):  # dfA1 ~ dfA31
    df = globals().get(f'dfA{i}')  # 변수 가져오기
    if df is not None:
        cols = set(df.columns)  # 컬럼명을 집합(set)으로 변환
        if common_columns_A is None:
            common_columns_A = cols  # 첫 번째 DataFrame의 컬럼을 초기값으로 설정
        else:
            common_columns_A &= cols  # 교집합 업데이트 (공통 컬럼만 남김)
# 결과 출력
print("공통으로 포함된 컬럼:", common_columns_A)
# 공통된 column 만 남기고 삭제 (A)
for i in range(1, 33):  # dfA1 ~ dfA33
    df = globals().get(f'dfA{i}')  # 변수 가져오기
    if df is not None:
        df_filtered_A = df[list(common_columns_A)]  # set을 list로 변환하여 사용
        globals()[f'dfA{i}'] = df_filtered_A  # 원래 변수에 다시 저장
# 라벨 칼럼 생성(A)
df_list = []
for i in range(1, 33):  # dfA1 ~ dfA32
    df_name = f"dfA{i}"  # DataFrame 이름 문자열 생성
    df = globals().get(df_name)  # 해당 변수 가져오기
    if df is not None:  # DataFrame이 존재하면 실행
        file_id = f"A{i}"  # "df"를 제외한 이름 생성
        df["file_name"] = file_id  # "file_name" 컬럼 추가
        df = df[["file_name"] + [col for col in df.columns if col != "file_name"]]  # 컬럼 순서 변경
        df_list.append(df)
        globals()[df_name] = df  # 변경된 DataFrame을 다시 저장
# 모든 dfA1 ~ dfA32를 하나로 합쳐서 dfA 생성
dfA = pd.concat(df_list, ignore_index=True)
dfA['file_name'].unique()

dfA = dfA[["Time [s]"] + [col for col in dfA.columns if col != "Time [s]"]]
dfA = dfA.drop(columns=['min. SoC [%]', 'max. SoC [%)', 'max. Battery Temperature [°C]'])
dfA

In [ ]:
# 이상치 확인
fig, axes = plt.subplots(4, 8, figsize=(20, 15))
axes = axes.flatten()

for i in range(1, 33):
    var_name = f"dfA{i}"
    sns.boxplot(y=globals()[var_name]["Battery Voltage [V]"], ax=axes[i-1], color="skyblue")
    axes[i-1].set_title(var_name, fontsize=12)
    axes[i-1].set_xlabel("")

for j in range(i, 32):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

- 이상치 존재 BUT 이상치와 정상범위의 값 차이가 크지 않고, 데이터 손실이 우려돼 삭제하지 않고 그냥 진행
- 중복행 존재 X

In [ ]:
# dfA에 SoC만 차분 적용
dfA_2=dfA.copy()
dfA_2["SoC_shift"] = dfA_2.groupby("file_name")["SoC [%]"].shift(1)
dfA_2["SoC_diff"] = dfA_2["SoC [%]"] - dfA_2["SoC_shift"]
dfA_2

In [ ]:
# SoC_diff vs 다른 데이터들

# file_name 리스트
file_names = [f"A{i}" for i in range(1, 33)]

# 개별 그래프 생성
for i, file_name in enumerate(file_names):
    df_subset = dfA_2[dfA_2["file_name"] == file_name]

    if not df_subset.empty:
        # X축 최소/최대 시간 값 계산
        min_time = int(df_subset["Time [s]"].min())
        max_time = int(df_subset["Time [s]"].max())

        # 기존 개별 y축 최소/최대 값 계산
        min_y1, max_y1 = df_subset["SoC_diff"].min(), df_subset["SoC_diff"].max()
        min_y2, max_y2 = df_subset["Motor Torque [Nm]"].min(), df_subset["Motor Torque [Nm]"].max()

        # 0 위치를 맞추기 위한 y축 하한/상한 조정
        y1_range = max(abs(min_y1), abs(max_y1))  # SoC_diff 기준
        y2_range = max(abs(min_y2), abs(max_y2))  # Motor Torque 기준

        # SoC_diff와 Motor Torque의 0 위치가 동일하도록 조정
        adj_min_y1, adj_max_y1 = -y1_range, y1_range
        adj_min_y2, adj_max_y2 = -y2_range, y2_range

        # 그래프 가로 크기 동적 조정 (최대 3164초 기준)
        width = max_time / 100
        height = 5  # 고정 높이

        # 개별 그래프 설정
        fig, ax = plt.subplots(figsize=(width, height))
        plt.title(f"dfA{i+1} - SoC_diff & Motor Torque [Nm]]", fontsize=14)

        # y축 그래프
        ax.plot(df_subset["Time [s]"], df_subset["SoC_diff"], 'g-', linewidth=1.5, label="SoC_diff")

        # 보조 y축 그래프
        ax2 = ax.twinx()
        ax2.plot(df_subset["Time [s]"], df_subset["Motor Torque [Nm]"], 'b-', linewidth=1.5, label="Motor Torque [Nm]")

        # Y축 레이블
        ax.set_ylabel("SoC_diff", color='g')
        ax2.set_ylabel("Motor Torque [Nm]", color='b')

        # 기존 y축 범위를 유지하면서 0 위치만 맞춤
        ax.set_ylim(adj_min_y1, adj_max_y1)
        ax2.set_ylim(adj_min_y2, adj_max_y2)

        # Y축 눈금 색상 조정
        ax.tick_params(axis='y', labelcolor='g')
        ax2.tick_params(axis='y', labelcolor='b')

        # X축 눈금 설정 (100초 단위)
        plt.xticks(range(min_time, max_time + 1, 100))

        # X축 범위 설정
        plt.xlim(min_time, max_time)

        ax.legend(loc="upper left")
        ax2.legend(loc="upper right")
        
        plt.xlabel("Time [s]")
        plt.show()

**선정 컬럼**
- Battery Current [A]: 0.55
- Motor Torque [Nm]: -0.42
- Longitudinal Acceleration [m/s^2]: -0.39
- Throttle [%]: -0.36
- Battery Voltage [V]: 0.31
- Velocity [km/h]: -0.22
- Regenerative Braking Signal: 0.23
- AirCon Power [kW]: -0.04
- Battery Temperature [°C]: -0.04
- Ambient Temperature [°C]: -0.04

In [ ]:
# Feature Engineering
dfA_2["Battery Power (W)"] = dfA_2["Battery Voltage [V]"] * dfA_2["Battery Current [A]"]
dfA_2["Battery Capacity (Wh)"] = (dfA_2["SoC [%]"] / 100) * 22600
dfA_2["Delta Battery Capacity (Wh)"] = dfA_2["Battery Capacity (Wh)"].diff()
dfA_2['Distance (km)'] = (dfA_2['Velocity [km/h]'] * dfA_2['Time [s]']) / 3600
dfA_2["Energy Consumption (Wh/km)"] = dfA_2["Delta Battery Capacity (Wh)"] / dfA_2["Distance (km)"].diff()

In [ ]:
selected_columns = [
    "Time [s]", "file_name", "Battery Current [A]", "Motor Torque [Nm]", 
    "Longitudinal Acceleration [m/s^2]", "Throttle [%]", "Battery Voltage [V]", 
    "Velocity [km/h]", "Regenerative Braking Signal ", "AirCon Power [kW]", 
    "Battery Temperature [°C]", "Ambient Temperature [°C]", 
    "SoC [%]", "SoC_diff", "Battery Power (W)", "Battery Capacity (Wh)", 
    "Delta Battery Capacity (Wh)", "Distance (km)", "Energy Consumption (Wh/km)"
]
dfA_3=dfA_2[selected_columns].copy()

In [ ]:
# SoC_diff와 새로 추가한 컬럼들 경향성 분석

# file_name 리스트
file_names = [f"A{i}" for i in range(1, 33)]

# 개별 그래프 생성
for i, file_name in enumerate(file_names):
    df_subset = dfA_2[dfA_2["file_name"] == file_name]

    if not df_subset.empty:
        # X축 최소/최대 시간 값 계산
        min_time = int(df_subset["Time [s]"].min())
        max_time = int(df_subset["Time [s]"].max())

        # 기존 개별 y축 최소/최대 값 계산
        min_y1, max_y1 = df_subset["SoC_diff"].min(), df_subset["SoC_diff"].max()
        min_y2, max_y2 = df_subset["Battery Capacity (Wh)"].min(), df_subset["Battery Capacity (Wh)"].max()

        # 0 위치를 맞추기 위한 y축 하한/상한 조정
        y1_range = max(abs(min_y1), abs(max_y1))  # SoC_diff 기준
        y2_range = max(abs(min_y2), abs(max_y2))  # Motor Torque 기준

        # SoC_diff와 Motor Torque의 0 위치가 동일하도록 조정
        adj_min_y1, adj_max_y1 = -y1_range, y1_range
        adj_min_y2, adj_max_y2 = -y2_range, y2_range

        # 그래프 가로 크기 동적 조정 (최대 3164초 기준)
        width = max_time / 100
        height = 5  # 고정 높이

        # 개별 그래프 설정
        fig, ax = plt.subplots(figsize=(width, height))
        plt.title(f"dfA{i+1} - SoC_diff & Battery Capacity (Wh)]", fontsize=14)

        # y축 그래프
        ax.plot(df_subset["Time [s]"], df_subset["SoC_diff"], 'g-', linewidth=1.5, label="SoC_diff")

        # 보조 y축 그래프
        ax2 = ax.twinx()
        ax2.plot(df_subset["Time [s]"], df_subset["Battery Capacity (Wh)"], 'b-', linewidth=1.5, label="Battery Capacity (Wh)")

        # Y축 레이블
        ax.set_ylabel("SoC_diff", color='g')
        ax2.set_ylabel("Battery Capacity (Wh)", color='b')

        # 기존 y축 범위를 유지하면서 0 위치만 맞춤
        ax.set_ylim(adj_min_y1, adj_max_y1)
        ax2.set_ylim(adj_min_y2, adj_max_y2)

        # Y축 눈금 색상 조정
        ax.tick_params(axis='y', labelcolor='g')
        ax2.tick_params(axis='y', labelcolor='b')

        # X축 눈금 설정 (100초 단위)
        plt.xticks(range(min_time, max_time + 1, 100))

        # X축 범위 설정
        plt.xlim(min_time, max_time)

        ax.legend(loc="upper left")
        ax2.legend(loc="upper right")
        
        plt.xlabel("Time [s]")
        plt.show()

**최종 선정 컬럼**
- Battery Current [A]
- Motor Torque [Nm]
- Longitudinal Acceleration [m/s^2]
- Throttle [%]
- Battery Voltage [V]
- Velocity [km/h]
- Regenerative Braking Signal
- AirCon Power [kW]
- Battery Temperature [°C]
- Ambient Temperature [°C]
- Battery Power (W)
- Battery Capacity (Wh)
- Delta Battery Capacity (Wh)
- Distance (km)
- Energy Consumption (Wh/km)

-----------

**B**

In [ ]:
# TripB01~TripB38까지
for i in range(1, 39):
    file_name = f"TripB{i:02d}.csv"
    globals()[f"dfB{i}"] = pd.read_csv(file_name, encoding="ISO-8859-1", delimiter=";")
    # Time [s] 컬럼을 datetime 형식으로 변환
    globals()[f"dfB{i}"]['Time [s]'] = pd.to_datetime(globals()[f"dfB{i}"]['Time [s]'], unit='s', origin='unix')
    # time 컬럼을 1초 단위로 묶기
    globals()[f"dfB{i}"] = globals()[f"dfB{i}"].set_index('Time [s]').resample('5S').mean().reset_index()
    # 필요시 Time [s] 컬럼 다시 생성
    globals()[f"dfB{i}"]['Time [s]'] = (globals()[f"dfB{i}"]['Time [s]'].astype(np.int64) // 10**9).astype(int)
    
# dfB1 ~ dfB38에 대해 선형 보간 적용
for i in range(1, 39):
    var_name = f"dfB{i}"
    new_var_name = f"dfB{i}_1"

    globals()[new_var_name] = globals()[var_name].interpolate(method='linear', limit_direction='both')
    

# dfB38_1 컬럼 명 수정
dfB38_1 = dfB38_1.rename(columns={"Velocity [km/h]]]": "Velocity [km/h]"})

# dfB_1로 통합

# 공통된 column 확인 (B)
import pandas as pd
# 공통 컬럼을 저장할 변수 (초기값을 None으로 설정)
common_columns_B = None
for i in range(1, 39):  # dfB1 ~ dfB38
    df = globals().get(f'dfB{i}_1')  # 변수 가져오기
    if df is not None:
        cols = set(df.columns)  # 컬럼명을 집합(set)으로 변환
        if common_columns_B is None:
            common_columns_B = cols  # 첫 번째 DataFrame의 컬럼을 초기값으로 설정
        else:
            common_columns_B &= cols  # 교집합 업데이트 (공통 컬럼만 남김)
# 결과 출력
print("공통으로 포함된 컬럼:", common_columns_B)
# 공통된 column 만 남기고 삭제 (B)
for i in range(1, 39):  # dfB1 ~ dfB38
    df = globals().get(f'dfB{i}_1')  # 변수 가져오기
    if df is not None:
        df_filtered_B = df[list(common_columns_B)]  # set을 list로 변환하여 사용
        globals()[f'dfB{i}_1'] = df_filtered_B  # 원래 변수에 다시 저장
# 라벨 칼럼 생성(B)
df_list = []
for i in range(1, 39):  # dfB1 ~ dfB38
    df_name = f"dfB{i}_1"  # DataFrame 이름 문자열 생성
    df = globals().get(df_name)  # 해당 변수 가져오기
    if df is not None:  # DataFrame이 존재하면 실행
        file_id = f"B{i}"  # "df"를 제외한 이름 생성
        df["file_name"] = file_id  # "file_name" 컬럼 추가
        df = df[["file_name"] + [col for col in df.columns if col != "file_name"]]  # 컬럼 순서 변경
        df_list.append(df)
        globals()[df_name] = df  # 변경된 DataFrame을 다시 저장
# 모든 dfB1 ~ dfB38를 하나로 합쳐서 dfB 생성
dfB_1 = pd.concat(df_list, ignore_index=True)
dfB_1 = dfB_1[["Time [s]"] + [col for col in dfB_1.columns if col != "Time [s]"]]
dfB_1 = dfB_1.drop(columns=['min. SoC [%]', 'max. SoC [%)', 'max. Battery Temperature [°C]', 'Coolant Volume Flow +500 [l/h]',
                            'Coolant Temperature Inlet [°C]', 'Coolant Temperature Heatercore [°C]', 'Requested Coolant Temperature [°C]'])
dfB_1

In [ ]:
# 차분
dfB_2=dfB_1.copy()
dfB_2["SoC_shift"] = dfB_2.groupby("file_name")["SoC [%]"].shift(1)
dfB_2["SoC_diff"] = dfB_2["SoC [%]"] - dfB_2["SoC_shift"]

# Feature Engeenring으로 컬럼 추가
dfB_2["Battery Power (W)"] = dfB_2["Battery Voltage [V]"] * dfB_2["Battery Current [A]"]
dfB_2["Battery Capacity (Wh)"] = (dfB_2["SoC [%]"] / 100) * 22600
dfB_2["Delta Battery Capacity (Wh)"] = dfB_2["Battery Capacity (Wh)"].diff()
dfB_2['Distance (km)'] = (dfB_2['Velocity [km/h]'] * dfB_2['Time [s]']) / 3600
dfB_2["Energy Consumption (Wh/km)"] = dfB_2["Delta Battery Capacity (Wh)"] / dfB_2["Distance (km)"].diff()
dfB_2['HVAC Power Consumption'] = dfB_2['Heating Power CAN [kW]']+dfB_2['AirCon Power [kW]']

In [ ]:
dfB_2_filtered = dfB_2.drop(columns=["Time [s]", "file_name"], errors="ignore")

# 피어슨 상관계수 계산
df_corr = dfB_2_filtered.corr()

# 하삼각형 마스크 생성
mask = np.triu(np.ones_like(df_corr, dtype=bool))

# 히트맵 그리기
plt.figure(figsize=(25, 25))
sns.heatmap(df_corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Lower Triangle Correlation Heatmap (Without Time [s] & file_name)")
plt.show()

In [ ]:
correlation_matrix = dfB_2_filtered.corr()[['SoC_diff']].drop(index='SoC_diff')

# 히트맵 그리기
plt.figure(figsize=(6, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, cbar=True)
plt.title('Correlation of SoC_diff with Other Variables')
plt.show()


In [ ]:
# SoC_diff vs 나머지 데이터

# file_name 리스트
file_names = [f"B{i}" for i in range(1, 39)]

# 개별 그래프 생성
for i, file_name in enumerate(file_names):
    df_subset = dfB_2[dfB_2["file_name"] == file_name]

    if not df_subset.empty:
        # X축 최소/최대 시간 값 계산
        min_time = int(df_subset["Time [s]"].min())
        max_time = int(df_subset["Time [s]"].max())

        # 기존 개별 y축 최소/최대 값 계산
        min_y1, max_y1 = df_subset["SoC_diff"].min(), df_subset["SoC_diff"].max()
        min_y2, max_y2 = df_subset["Motor Torque [Nm]"].min(), df_subset["Motor Torque [Nm]"].max()

        # 0 위치를 맞추기 위한 y축 하한/상한 조정
        y1_range = max(abs(min_y1), abs(max_y1))  # SoC_diff 기준
        y2_range = max(abs(min_y2), abs(max_y2))  # Ambient Temperature 기준

        # SoC_diff와 Ambient Temperature의 0 위치가 동일하도록 조정
        adj_min_y1, adj_max_y1 = -y1_range, y1_range
        adj_min_y2, adj_max_y2 = -y2_range, y2_range

        # 그래프 가로 크기 동적 조정 (최대 3164초 기준)
        width = max_time / 100
        height = 5  # 고정 높이

        # 개별 그래프 설정
        fig, ax = plt.subplots(figsize=(width, height))
        plt.title(f"dfB{i+1} - SoC_diff & Motor Torque [Nm]", fontsize=14)

        # y축 그래프
        ax.plot(df_subset["Time [s]"], df_subset["SoC_diff"], 'g-', linewidth=1.5, label="SoC_diff")

        # 보조 y축 그래프
        ax2 = ax.twinx()
        ax2.plot(df_subset["Time [s]"], df_subset["Motor Torque [Nm]"], 'b-', linewidth=1.5, label="Motor Torque [Nm]")

        # Y축 레이블
        ax.set_ylabel("SoC_diff", color='g')
        ax2.set_ylabel("Motor Torque [Nm]", color='b')

        # 기존 y축 범위를 유지하면서 0 위치만 맞춤
        ax.set_ylim(adj_min_y1, adj_max_y1)
        ax2.set_ylim(adj_min_y2, adj_max_y2)

        # Y축 눈금 색상 조정
        ax.tick_params(axis='y', labelcolor='g')
        ax2.tick_params(axis='y', labelcolor='b')

        # X축 눈금 설정 (100초 단위)
        plt.xticks(range(min_time, max_time + 1, 100))

        # X축 범위 설정
        plt.xlim(min_time, max_time)

        ax.legend(loc="upper left")
        ax2.legend(loc="upper right")
        
        plt.xlabel("Time [s]")
        plt.show()

**최종 선정 컬럼**
- Battery Voltage [V]
- Throttle [%]
- Battery Power (W)
- Battery Current [A]
- Heater Voltage [V]
- Requested Heating Power [W]
- Motor Torque [Nm]
- Delta Battery Capacity (Wh)
- Distance (km)
- Longitudinal Acceleration [m/s^2]
- Velocity [km/h]
- Regenerative Braking Signal
- Battery Temperature [°C]
- Ambient Temperature [°C]
- Battery Capacity (Wh)
- AirCon Power [kW]

-------------

SoC vs 정지했을 때 비교

In [ ]:
import matplotlib.pyplot as plt

# file_name 리스트
file_names = [f"A{i}" for i in range(1, 33)]

# 개별 그래프 생성
for i, file_name in enumerate(file_names):
    df_subset = dfA_2[dfA_2["file_name"] == file_name]

    if not df_subset.empty:
        # X축 최소/최대 시간 값 계산
        min_time = int(df_subset["Time [s]"].min())
        max_time = int(df_subset["Time [s]"].max())

        # 기존 개별 y축 최소/최대 값 계산
        min_y1, max_y1 = df_subset["SoC [%]"].min(), df_subset["SoC [%]"].max()
        min_y2, max_y2 = df_subset["Velocity [km/h]"].min(), df_subset["Velocity [km/h]"].max()

        # 0 위치를 맞추기 위한 y축 조정
        y1_range = max(abs(min_y1), abs(max_y1))  # SoC [%] 기준
        y2_range = max_y2 - min_y2  # Velocity [km/h] 범위

        # SoC [%]와 Velocity [km/h]의 0 위치 맞춤
        adj_min_y1, adj_max_y1 = -y1_range, y1_range
        adj_min_y2, adj_max_y2 = min_y2, min_y2 + y2_range  # 최소값 기준으로 조정

        # 그래프 가로 크기 동적 조정 (최대 3164초 기준)
        width = max(8, max_time / 100)  # 최소 8 이상의 크기로 설정
        height = 5  # 고정 높이

        # 개별 그래프 설정
        fig, ax = plt.subplots(figsize=(width, height))
        plt.title(f"dfA{i+1} - SoC [%] & Velocity [km/h]", fontsize=14)

        # SoC [%] 그래프 (녹색 선)
        ax.plot(df_subset["Time [s]"], df_subset["SoC [%]"], 'g-', linewidth=1.5, label="SoC [%]")

        # Velocity [km/h] 그래프 (파란색 선)
        ax2 = ax.twinx()
        ax2.plot(df_subset["Time [s]"], df_subset["Velocity [km/h]"], 'b-', linewidth=1.5, label="Velocity [km/h]")

        # Y축 레이블 설정
        ax.set_ylabel("SoC [%]", color='g')
        ax2.set_ylabel("Velocity [km/h]", color='b')

        # 기존 y축 범위를 유지하면서 0 위치 맞춤
        ax.set_ylim(adj_min_y1, adj_max_y1)
        ax2.set_ylim(adj_min_y2, adj_max_y2)  # 최소 속도부터 표시

        # Y축 눈금 색상 조정
        ax.tick_params(axis='y', labelcolor='g')
        ax2.tick_params(axis='y', labelcolor='b')

        # X축 눈금 설정 (가독성을 위해 동적 설정)
        xtick_step = max(100, (max_time - min_time) // 10)  # 최소 100초 간격
        plt.xticks(range(min_time, max_time + 1, xtick_step))

        # X축 범위 설정
        plt.xlim(min_time, max_time)

        ax.legend(loc="upper left")
        ax2.legend(loc="upper right")

        plt.xlabel("Time [s]")
        plt.show()

In [ ]:
import matplotlib.pyplot as plt

# file_name 리스트
file_names = [f"B{i}" for i in range(1, 39)]

# 개별 그래프 생성
for i, file_name in enumerate(file_names):
    df_subset = dfB_2[dfB_2["file_name"] == file_name]

    if not df_subset.empty:
        # X축 최소/최대 시간 값 계산
        min_time = int(df_subset["Time [s]"].min())
        max_time = int(df_subset["Time [s]"].max())

        # 기존 개별 y축 최소/최대 값 계산
        min_y1, max_y1 = df_subset["SoC [%]"].min(), df_subset["SoC [%]"].max()
        min_y2, max_y2 = df_subset["Velocity [km/h]"].min(), df_subset["Velocity [km/h]"].max()

        # 0 위치를 맞추기 위한 y축 조정
        y1_range = max(abs(min_y1), abs(max_y1))  # SoC [%] 기준
        y2_range = max_y2 - min_y2  # Velocity [km/h] 범위

        # SoC [%]와 Velocity [km/h]의 0 위치 맞춤
        adj_min_y1, adj_max_y1 = -y1_range, y1_range
        adj_min_y2, adj_max_y2 = min_y2, min_y2 + y2_range  # 최소값 기준으로 조정

        # 그래프 가로 크기 동적 조정 (최대 3164초 기준)
        width = max(8, max_time / 100)  # 최소 8 이상의 크기로 설정
        height = 5  # 고정 높이

        # 개별 그래프 설정
        fig, ax = plt.subplots(figsize=(width, height))
        plt.title(f"dfB{i+1} - SoC [%] & Velocity [km/h]", fontsize=14)

        # SoC [%] 그래프 (녹색 선)
        ax.plot(df_subset["Time [s]"], df_subset["SoC [%]"], 'g-', linewidth=1.5, label="SoC [%]")

        # Velocity [km/h] 그래프 (파란색 선)
        ax2 = ax.twinx()
        ax2.plot(df_subset["Time [s]"], df_subset["Velocity [km/h]"], 'b-', linewidth=1.5, label="Velocity [km/h]")

        # Y축 레이블 설정
        ax.set_ylabel("SoC [%]", color='g')
        ax2.set_ylabel("Velocity [km/h]", color='b')

        # 기존 y축 범위를 유지하면서 0 위치 맞춤
        ax.set_ylim(adj_min_y1, adj_max_y1)
        ax2.set_ylim(adj_min_y2, adj_max_y2)  # 최소 속도부터 표시

        # Y축 눈금 색상 조정
        ax.tick_params(axis='y', labelcolor='g')
        ax2.tick_params(axis='y', labelcolor='b')

        # X축 눈금 설정 (가독성을 위해 동적 설정)
        xtick_step = max(100, (max_time - min_time) // 10)  # 최소 100초 간격
        plt.xticks(range(min_time, max_time + 1, xtick_step))

        # X축 범위 설정
        plt.xlim(min_time, max_time)

        ax.legend(loc="upper left")
        ax2.legend(loc="upper right")

        plt.xlabel("Time [s]")
        plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 공통된 컬럼 찾기
common_columns = list(set(dfA_2.columns) & set(dfB_2.columns))

# 스타일 설정
sns.set_style("whitegrid")

# 히스토그램 그리기 (세로형 레이아웃)
num_cols = len(common_columns)
fig, axes = plt.subplots(nrows=num_cols, ncols=1, figsize=(10, 4 * num_cols))  # 한 줄에 하나씩

if num_cols == 1:
    axes = [axes]  # 단일 그래프 처리

for i, col in enumerate(common_columns):
    ax = axes[i]
    sns.histplot(dfA_2[col], bins=50, color="red", kde=True, label="dfA_2", ax=ax)
    sns.histplot(dfB_2[col], bins=50, color="blue", kde=True, label="dfB_2", ax=ax)
    ax.set_title(f"Histogram of {col}", fontsize=14)
    ax.legend()

plt.tight_layout()
plt.show()

## 온도 별 주행거리 산점도

In [ ]:
d[d['Route/Area']=='Munich East']

In [ ]:
d['Battery Temp diff'] = d['Battery Temperature (End)']-d['Battery Temperature (Start) [°C]']
d['Mean Battery Temperature'] = (d['Battery Temperature (Start) [°C]']+d['Battery Temperature (End)'])/2
d['Battery Capacity Consumption'] = (d['Battery State of Charge (Start)']-d['Battery State of Charge (End)'])*226
d['Battery Efficiency'] = (d['Battery State of Charge (Start)']-d['Battery State of Charge (End)'])*226/d['Distance [km]']

In [ ]:
d['Route/Area'].unique()

## 동일 코스임에도 온도에 따른 배터리 용량 소모 차이가 있는지

## 주행 코스 구분 없이 온도 vs 배터리 용량 소모

In [ ]:
import matplotlib.pyplot as plt

# 산점도 그리기
plt.figure(figsize=(8, 6))
plt.scatter(d['Mean Battery Temperature'], d['Battery Efficiency'], alpha=0.5)
plt.xlabel('Mean Battery Temperature')
plt.ylabel('Battery Efficiency')
plt.title('Scatter Plot of Battery Efficiency vs. Mean Battery Temperature')
plt.grid(True)
plt.show()

## 짐 무게에 따른 배터리 용량 소모 비교

In [ ]:
import matplotlib.pyplot as plt

# TripB22, TripB24, TripB25, TripB34 데이터 필터링
trip_data = d[d['Trip'].isin(['TripB22', 'TripB24', 'TripB25', 'TripB34'])]

# 각 Trip의 Battery Efficiency 평균 계산
efficiency_means = trip_data.groupby('Trip')['Battery Efficiency'].mean()

# 원하는 순서로 정렬
desired_order = ['TripB24', 'TripB25', 'TripB22', 'TripB34']
efficiency_means = efficiency_means.reindex(desired_order)

# 바 차트 그리기
plt.figure(figsize=(6, 5))
efficiency_means.plot(kind='bar', color=['blue', 'orange', 'blue', 'orange'], alpha=0.7)

plt.xlabel('Trip')
plt.ylabel('Battery Efficiency')
plt.title('Comparison of Battery Efficiency by weight')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

## 배터리 온도 변화에 따른 배터리 효율 변화

In [ ]:
import matplotlib.pyplot as plt


# 산점도 그리기
plt.figure(figsize=(8, 6))
plt.scatter(d['Battery Temp diff'], d['Battery Efficiency'], alpha=0.5)
plt.xlabel('Battery Temperature difference')
plt.ylabel('Battery Efficiency')
plt.title('Scatter Plot of Battery Temperature difference vs. Battery Efficiency')
plt.grid(True)
plt.show()

------------

이전 데이터셋 리셋하고 진행: 그래프에서 경향성을 보다 쉽게 파악하기 위해 5초 단위로 묶었으나, 밑에서부터는 롤링으로 진행할거라 리셋

In [ ]:
# 이 전 내용 리셋
from IPython.display import clear_output
from IPython import get_ipython

# 출력 지우기 (reset 실행 전에)
clear_output()

# 모든 변수 및 실행된 내용 초기화
get_ipython().run_line_magic("reset", "-f")

print("Jupyter Notebook 환경이 초기화됐습니다.")

-----------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [ ]:
d=pd.read_excel('Overview.xlsx')

In [ ]:
# TripA01~TripA32까지
for i in range(1, 33):
    file_name = f"TripA{i:02d}.csv"
    globals()[f"dfA{i}"] = pd.read_csv(file_name, encoding="ISO-8859-1", delimiter=";")

    # Time [s] 컬럼을 datetime 형식으로 변환
    # globals()[f"dfA{i}"]['Time [s]'] = pd.to_datetime(globals()[f"dfA{i}"]['Time [s]'], unit='s', origin='unix')

# dfA11 결측치 제거
dfA11=dfA11.drop(columns=["Unnamed: 23"], errors="ignore")

# dfA로 통합

# 공통된 column 확인 (A)
# 공통 컬럼을 저장할 변수 (초기값을 None으로 설정)
common_columns_A = None
for i in range(1, 33):  # dfA1 ~ dfA31
    df = globals().get(f'dfA{i}')  # 변수 가져오기
    if df is not None:
        cols = set(df.columns)  # 컬럼명을 집합(set)으로 변환
        if common_columns_A is None:
            common_columns_A = cols  # 첫 번째 DataFrame의 컬럼을 초기값으로 설정
        else:
            common_columns_A &= cols  # 교집합 업데이트 (공통 컬럼만 남김)
# 결과 출력
print("공통으로 포함된 컬럼:", common_columns_A)
# 공통된 column 만 남기고 삭제 (A)
for i in range(1, 33):  # dfA1 ~ dfA32
    df = globals().get(f'dfA{i}')  # 변수 가져오기
    if df is not None:
        df_filtered_A = df[list(common_columns_A)]  # set을 list로 변환하여 사용
        globals()[f'dfA{i}'] = df_filtered_A  # 원래 변수에 다시 저장
# 라벨 칼럼 생성(A)
df_list = []
for i in range(1, 33):  # dfB1 ~ dfA32
    df_name = f"dfA{i}"  # DataFrame 이름 문자열 생성
    df = globals().get(df_name)  # 해당 변수 가져오기
    if df is not None:  # DataFrame이 존재하면 실행
        file_id = f"A{i}"  # "df"를 제외한 이름 생성
        df["file_name"] = file_id  # "file_name" 컬럼 추가
        df = df[["file_name"] + [col for col in df.columns if col != "file_name"]]  # 컬럼 순서 변경
        df_list.append(df)
        globals()[df_name] = df  # 변경된 DataFrame을 다시 저장
# 모든 dfA1 ~ dfA32를 하나로 합쳐서 dfA 생성
dfA = pd.concat(df_list, ignore_index=True)
dfA['file_name'].unique()

dfA = dfA[["Time [s]"] + [col for col in dfA.columns if col != "Time [s]"]]

dfA = dfA.drop(columns=['min. SoC [%]', 'max. SoC [%)', 'max. Battery Temperature [°C]'])

#SoC만 차분 적용
dfA_2=dfA.copy()
dfA_2["SoC_shift"] = dfA_2.groupby("file_name")["SoC [%]"].shift(1)
dfA_2["SoC_diff"] = dfA_2["SoC [%]"] - dfA_2["SoC_shift"]

#피쳐 엔지니어링
dfA_2["Battery Power (W)"] = dfA_2["Battery Voltage [V]"] * dfA_2["Battery Current [A]"]
##머신러닝용
dfA_2['Distance (km)'] = (dfA_2['Velocity [km/h]'] * dfA_2['Time [s]']) / 36000
dfA_2["Distance (km)_shift"] = dfA_2.groupby("file_name")["Distance (km)"].shift(1)
dfA_2["Battery Capacity (Wh)_shift"] = (dfA_2["SoC_shift"] / 100) * 22600
dfA_2["Delta Battery Capacity (Wh)_shift"] = dfA_2.groupby("file_name")["Battery Capacity (Wh)_shift"].diff()
dfA_2["Energy Consumption (Wh/km)_shift"] = dfA_2.groupby("file_name")["Delta Battery Capacity (Wh)_shift"].transform(lambda x: x / dfA_2["Distance (km)_shift"].diff())
##대시보드용
dfA_2["Battery Capacity (Wh)"] = (dfA_2["SoC [%]"] / 100) * 22600
dfA_2["Delta Battery Capacity (Wh)"] = dfA_2.groupby("file_name")["Battery Capacity (Wh)"].diff()
dfA_2["Energy Consumption (Wh/km)"] = dfA_2.groupby("file_name")["Delta Battery Capacity (Wh)"].transform(lambda x: x / dfA_2["Distance (km)"].diff())

#추가된 컬럼
dfA_2['HVAC Power Consumption'] = dfA_2['Heating Power CAN [kW]']+dfA_2['AirCon Power [kW]']
##불필요 칼럼 제외
dfA_2 = dfA_2.drop("Distance (km)_shift",axis=1)

#5초 롤링
dfA_5 = pd.DataFrame()
for col in dfA_2.columns:
    if col not in ["file_name", "Time [s]", "Battery Capacity (Wh)","Delta Battery Capacity (Wh)","Energy Consumption (Wh/km)"]:  # 파일명,시간,대시보드용 제외
        dfA_5[col] = dfA_2[col].rolling(window=50, min_periods=1).mean()  # 롤링 평균 적용

In [ ]:
# dfa_5에 Time [s]추가 + 맨 앞으로 정렬
dfA_5['Time [s]']=dfA_2['Time [s]']
cols = ["Time [s]"] + [col for col in dfA_5.columns if col != "Time [s]"]
dfA_5 = dfA_5[cols]

In [ ]:
dfA_5.info()

In [ ]:
# 5초 머신러닝
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 예측에 사용할 X, y 정의
list_5_A=['Battery Current [A]', 'Battery Power (W)', 'HVAC Power Consumption', 'Battery Capacity (Wh)_shift', 'Velocity [km/h]', 'Battery Voltage [V]',
          'Throttle [%]', 'AirCon Power [kW]', 'Ambient Temperature [°C]', 'Longitudinal Acceleration [m/s^2]', 'Motor Torque [Nm]', 'Battery Temperature [°C]',
          'Regenerative Braking Signal ']
dfA_5_clean = dfA_5.dropna()  # 결측값 제거
X = dfA_5_clean[list_5_A]
y = dfA_5_clean['SoC_diff']

# 특수문자 제거한 새로운 컬럼 이름 리스트 생성
X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습 및 평가
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

plt.figure(figsize=(12, 5))

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {"MSE": mse, "R²": r2, "Feature Importance": model.feature_importances_, "y_pred": y_pred}
    
    print(f"{name} 모델 성능:")
    print(f"  - MSE: {mse:.4f}")
    print(f"  - R²: {r2:.4f}\n")

    # 피처 중요도 시각화
    plt.subplot(1, 2, i + 1)
    sorted_idx = np.argsort(results[name]["Feature Importance"])
    plt.barh(X_train.columns[sorted_idx], results[name]["Feature Importance"][sorted_idx])
    plt.xlabel("Feature Importance")
    plt.title(f"{name} Feature Importance")

plt.tight_layout()
plt.show()

# 예측값 vs 실제값 비교 그래프 (산점도)
plt.figure(figsize=(12, 5))
for i, (name, result) in enumerate(results.items()):
    plt.subplot(1, 2, i + 1)
    plt.scatter(y_test, result["y_pred"], alpha=0.5)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color="red", linestyle="--")  # 완벽한 예측선
    plt.xlabel("Actual SoC_diff")
    plt.ylabel("Predicted SoC_diff")
    plt.title(f"{name} Prediction vs Actual")
    plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
# 과적합 확인
for name, model in models.items():
    # 훈련 데이터 예측 및 평가
    y_train_pred = model.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    
    # 테스트 데이터 예측 및 평가
    y_test_pred = model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"{name} 모델 성능 비교:")
    print(f"  - 훈련 데이터 R²: {train_r2:.4f}")
    print(f"  - 테스트 데이터 R²: {test_r2:.4f}")
    
    # 과적합 판단 기준
    if train_r2 > 0.9 and (train_r2 - test_r2) > 0.1:
        print("과적합이 의심됩니다.")
    else:
        print("과적합이 의심되지 않습니다.")
    
    print("\n")

-------------

In [ ]:
# TripB01~TripB38까지
for i in range(1, 39):
    file_name = f"TripB{i:02d}.csv"
    globals()[f"dfB{i}"] = pd.read_csv(file_name, encoding="ISO-8859-1", delimiter=";")

# dfB38 컬럼 명 수정
dfB38 = dfB38.rename(columns={"Velocity [km/h]]]": "Velocity [km/h]"})

# dfB1 ~ dfB38에서 해당 컬럼 제거
for i in range(1, 39):
    var_name = f"dfB{i}"
    
    if "Coolant Volume Flow +500 [l/h]" in globals()[var_name].columns:
        globals()[var_name] = globals()[var_name].drop(columns=["Coolant Volume Flow +500 [l/h]"])

# dfB1 ~ dfB38에 대해 선형 보간 적용
for i in range(1, 39):
    var_name = f"dfB{i}"
    new_var_name = f"dfB{i}_1"

    globals()[new_var_name] = globals()[var_name].interpolate(method='linear', limit_direction='both')

# dfB로 통합

# 공통된 column 확인 (B)
# 공통 컬럼을 저장할 변수 (초기값을 None으로 설정)
common_columns_B = None
for i in range(1, 39):  # dfB1 ~ dfB38
    df = globals().get(f'dfB{i}_1')  # 변수 가져오기
    if df is not None:
        cols = set(df.columns)  # 컬럼명을 집합(set)으로 변환
        if common_columns_B is None:
            common_columns_B = cols  # 첫 번째 DataFrame의 컬럼을 초기값으로 설정
        else:
            common_columns_B &= cols  # 교집합 업데이트 (공통 컬럼만 남김)
# 결과 출력
print("공통으로 포함된 컬럼:", common_columns_B)
# 공통된 column 만 남기고 삭제 (B)
for i in range(1, 39):  # dfB1 ~ dfB38
    df = globals().get(f'dfB{i}_1')  # 변수 가져오기
    if df is not None:
        df_filtered_B = df[list(common_columns_B)]  # set을 list로 변환하여 사용
        globals()[f'dfB{i}_1'] = df_filtered_B  # 원래 변수에 다시 저장
# 라벨 칼럼 생성(B)
df_list = []
for i in range(1, 39):  # dfB1 ~ dfB38
    df_name = f"dfB{i}_1"  # DataFrame 이름 문자열 생성
    df = globals().get(df_name)  # 해당 변수 가져오기
    if df is not None:  # DataFrame이 존재하면 실행
        file_id = f"B{i}"  # "df"를 제외한 이름 생성
        df["file_name"] = file_id  # "file_name" 컬럼 추가
        df = df[["file_name"] + [col for col in df.columns if col != "file_name"]]  # 컬럼 순서 변경
        df_list.append(df)
        globals()[df_name] = df  # 변경된 DataFrame을 다시 저장
# 모든 dfB1 ~ dfB38를 하나로 합쳐서 dfB 생성
dfB = pd.concat(df_list, ignore_index=True)

dfB = dfB[["Time [s]"] + [col for col in dfB.columns if col != "Time [s]"]]

dfB = dfB.drop(columns=['min. SoC [%]', 'max. SoC [%)', 'max. Battery Temperature [°C]'])
dfB = dfB[dfB["file_name"] != "B4"]
dfB

In [ ]:
#SoC만 차분 적용
dfB_1=dfB.copy()
dfB_1["SoC_shift"] = dfB_1.groupby("file_name")["SoC [%]"].shift(1)
dfB_1["SoC_diff"] = dfB_1["SoC [%]"] - dfB_1["SoC_shift"]

#피쳐 엔지니어링
dfB_1["Battery Power (W)"] = dfB_1["Battery Voltage [V]"] * dfB_1["Battery Current [A]"]
##머신러닝용
dfB_1['Distance (km)'] = (dfB_1['Velocity [km/h]'] * dfB_1['Time [s]']) / 36000
dfB_1["Distance (km)_shift"] = dfB_1.groupby("file_name")["Distance (km)"].shift(1)
dfB_1["Battery Capacity (Wh)_shift"] = (dfB_1["SoC_shift"] / 100) * 22600
dfB_1["Delta Battery Capacity (Wh)_shift"] = dfB_1.groupby("file_name")["Battery Capacity (Wh)_shift"].diff()
dfB_1["Energy Consumption (Wh/km)_shift"] = dfB_1.groupby("file_name")["Delta Battery Capacity (Wh)_shift"].transform(lambda x: x / dfB_1["Distance (km)_shift"].diff())
##대시보드용
dfB_1["Battery Capacity (Wh)"] = (dfB_1["SoC [%]"] / 100) * 22600
dfB_1["Delta Battery Capacity (Wh)"] = dfB_1.groupby("file_name")["Battery Capacity (Wh)"].diff()
dfB_1["Energy Consumption (Wh/km)"] = dfB_1.groupby("file_name")["Delta Battery Capacity (Wh)"].transform(lambda x: x / dfB_1["Distance (km)"].diff())

# 추가 컬럼
dfB_1['HVAC Power Consumption'] = dfB_1['Heating Power CAN [kW]']+dfB_1['AirCon Power [kW]']

##불필요 칼럼 제외
dfB_1 = dfB_1.drop("Distance (km)_shift",axis=1)

#5초 롤링
dfB_5 = pd.DataFrame()
for col in dfB_1.columns:
    if col not in ["file_name", "Time [s]", "Battery Capacity (Wh)","Delta Battery Capacity (Wh)","Energy Consumption (Wh/km)"]:  # 파일명,시간,대시보드용 제외
        dfB_5[col] = dfB_1[col].rolling(window=50, min_periods=1).mean()  # 롤링 평균 적용

In [ ]:
# dfa_100에 Time [s]추가 + 맨 앞으로 정렬
dfB_5['Time [s]']=dfB_1['Time [s]']
cols = ["Time [s]"] + [col for col in dfB_5.columns if col != "Time [s]"]
dfB_5 = dfB_5[cols]

In [ ]:
dfB_5.info()

In [ ]:
# 5초
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 예측에 사용할 X, y 정의
list_5_B=['Battery Current [A]', 'Battery Power (W)', 'HVAC Power Consumption', 'Battery Capacity (Wh)_shift', 'Velocity [km/h]', 'Battery Voltage [V]', 'Throttle [%]',
        'Ambient Temperature [°C]', 'Longitudinal Acceleration [m/s^2]', 'Motor Torque [Nm]', 'Battery Temperature [°C]', 'AirCon Power [kW]',
        'Regenerative Braking Signal ', 'Requested Heating Power [W]', 'Heater Voltage [V]']
dfB_5_clean = dfB_5.dropna()  # 결측값 제거
X = dfB_5_clean[list_5_B]
y = dfB_5_clean['SoC_diff']

# 특수문자 제거한 새로운 컬럼 이름 리스트 생성
X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 모델 학습 및 평가
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

plt.figure(figsize=(12, 5))

for i, (name, model) in enumerate(models.items()):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {"MSE": mse, "R²": r2, "Feature Importance": model.feature_importances_, "y_pred": y_pred}
    
    print(f"{name} 모델 성능:")
    print(f"  - MSE: {mse:.4f}")
    print(f"  - R²: {r2:.4f}\n")

    # 피처 중요도 시각화
    plt.subplot(1, 2, i + 1)
    sorted_idx = np.argsort(results[name]["Feature Importance"])
    plt.barh(X_train.columns[sorted_idx], results[name]["Feature Importance"][sorted_idx])
    plt.xlabel("Feature Importance")
    plt.title(f"{name} Feature Importance")

plt.tight_layout()
plt.show()

# 예측값 vs 실제값 비교 그래프 (산점도)
plt.figure(figsize=(12, 5))
for i, (name, result) in enumerate(results.items()):
    plt.subplot(1, 2, i + 1)
    plt.scatter(y_test, result["y_pred"], alpha=0.5)
    plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color="red", linestyle="--")  # 완벽한 예측선
    plt.xlabel("Actual SoC_diff")
    plt.ylabel("Predicted SoC_diff")
    plt.title(f"{name} Prediction vs Actual")
    plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
# 과적합 확인
for name, model in models.items():
    # 훈련 데이터 예측 및 평가
    y_train_pred = model.predict(X_train)
    train_r2 = r2_score(y_train, y_train_pred)
    
    # 테스트 데이터 예측 및 평가
    y_test_pred = model.predict(X_test)
    test_r2 = r2_score(y_test, y_test_pred)

    print(f"{name} 모델 성능 비교:")
    print(f"  - 훈련 데이터 R²: {train_r2:.4f}")
    print(f"  - 테스트 데이터 R²: {test_r2:.4f}")
    
    # 과적합 판단 기준
    if train_r2 > 0.9 and (train_r2 - test_r2) > 0.1:
        print("과정합이 의심됩니다.")
    else:
        print("과적합이 의심되지 않습니다.")
    
    print("\n")

-----------

추가 자료

In [ ]:
# 예측에 사용할 X, y 정의
list_5_A=['Battery Current [A]', 'Battery Power (W)', 'HVAC Power Consumption', 'Battery Capacity (Wh)_shift', 'Velocity [km/h]', 'Battery Voltage [V]',
          'Throttle [%]', 'AirCon Power [kW]', 'Ambient Temperature [°C]', 'Longitudinal Acceleration [m/s^2]', 'Motor Torque [Nm]', 'Battery Temperature [°C]',
          'Regenerative Braking Signal ']
dfA_5_clean = dfA_5.dropna()  # 결측값 제거
X = dfA_5_clean[list_5_A]
y = dfA_5_clean['SoC_diff']

# 특수문자 제거한 새로운 컬럼 이름 리스트 생성
X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGB 모델 학습 및 평가
model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("XGBoost 모델 성능:")
print(f"  - MSE: {mse:.4f}")
print(f"  - R²: {r2:.4f}\n")

# 피처 중요도 시각화
plt.figure(figsize=(8, 5))
sorted_idx = np.argsort(model.feature_importances_)
plt.barh(X_train.columns[sorted_idx], model.feature_importances_[sorted_idx])
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.show()

# 예측값 vs 실제값 비교 그래프 (산점도)
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color="red", linestyle="--")  # 완벽한 예측선
plt.xlabel("Actual SoC_diff")
plt.ylabel("Predicted SoC_diff")
plt.title("XGBoost Prediction vs Actual")
plt.grid()
plt.show()

In [ ]:
# 예측에 사용할 X, y 정의
list_5_B=['Battery Current [A]', 'Battery Power (W)', 'HVAC Power Consumption', 'Battery Capacity (Wh)_shift', 'Velocity [km/h]', 'Battery Voltage [V]', 'Throttle [%]',
        'Ambient Temperature [°C]', 'Longitudinal Acceleration [m/s^2]', 'Motor Torque [Nm]', 'Battery Temperature [°C]', 'AirCon Power [kW]',
        'Regenerative Braking Signal ', 'Requested Heating Power [W]', 'Heater Voltage [V]']
dfB_5_clean = dfB_5.dropna()  # 결측값 제거
X = dfB_5_clean[list_5_B]
y = dfB_5_clean['SoC_diff']

# 특수문자 제거한 새로운 컬럼 이름 리스트 생성
X.columns = X.columns.str.replace(r"[\[\]<>]", "", regex=True)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGB 모델 학습 및 평가
model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("XGBoost 모델 성능:")
print(f"  - MSE: {mse:.4f}")
print(f"  - R²: {r2:.4f}\n")

# 피처 중요도 시각화
plt.figure(figsize=(8, 5))
sorted_idx = np.argsort(model.feature_importances_)
plt.barh(X_train.columns[sorted_idx], model.feature_importances_[sorted_idx])
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.show()

# 예측값 vs 실제값 비교 그래프 (산점도)
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color="red", linestyle="--")  # 완벽한 예측선
plt.xlabel("Actual SoC_diff")
plt.ylabel("Predicted SoC_diff")
plt.title("XGBoost Prediction vs Actual")
plt.grid()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
import pandas as pd

hvac_A = dfA_5['HVAC Power Consumption'].dropna()
hvac_B = dfB_5['HVAC Power Consumption'].dropna()

# Mann-Whitney U 검정 수행
u_stat, p_value = mannwhitneyu(hvac_A, hvac_B, alternative='two-sided')

# 결과 출력
print(f"Mann-Whitney U Statistic: {u_stat}")
print(f"p-value: {p_value}")

# 유의수준 0.05 기준으로 해석
if p_value < 0.05:
    print("두 그룹 간 차이가 통계적으로 유의합니다.")
else:
    print("두 그룹 간 차이가 통계적으로 유의하지 않습니다.")

In [ ]:
sns.lineplot(x='Time [s]', y='SoC [%]', data = dfB4)

In [ ]:
dfA_5_filtered = dfA_5.drop(columns=["Time [s]", "file_name"], errors="ignore")

df_corr = dfA_5_filtered.corr()
plt.figure(figsize=(13, 13))
sns.heatmap(df_corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap (Without Time [s] & file_name)")
plt.show()

In [ ]:
correlation_matrix = dfA_5_filtered.corr()[['SoC_diff']].drop(index='SoC_diff')

plt.figure(figsize=(5, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, cbar=True)
plt.title('Correlation of SoC_diff with Other Variables')
plt.show()